# Ground-Truth Labeled Overlay Generator

This notebook builds **definitive labeled overlays** for the Hyperspectral Ground-Truth Explorer dashboard using the ground-truth masks published in the source papers.

**Papers wired up:**
- HAD-100 — https://zhaoxuli123.github.io/HAD100/
- WHU-Hi — https://rsidea.whu.edu.cn/resource_WHUHiriver_sharing.htmz
- MUUFL Gulfport — https://github.com/GatorSense/MUUFLGulfport
- Zenodo #13370800 — https://zenodo.org/records/13370800
- HuggingFace HSI_Datasets — https://huggingface.co/datasets/Tanishq165/HSI_Datasets

**Output:** each scene with a GT mask gets a `labeled_thumb` overlay where every class gets a distinct color, the legend is baked into the image, and the whole payload is rebuilt as `payload_v4_labeled.json`. Drop that file into the dashboard repo, rebake `index.html`, push, and every card shows the true ground-truth labels.

**Runtime:** ~4-8 minutes on a CPU Colab. No GPU needed.

## 1. Setup — mount Drive and install deps

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q scipy h5py Pillow numpy matplotlib
print('ready')

## 2. Configure paths

Set `DRIVE_ROOT` to your shared folder (default: the Allotrope 21 Labs `hyperspectral_datasets` folder). Set `PAYLOAD_URL` to the current `payload_v3.json` (baked into `index.html`) — or upload one and set its local path.

In [ ]:
DRIVE_ROOT   = '/content/drive/MyDrive/hyperspectral_datasets'   # <- your Drive folder
PAYLOAD_IN   = '/content/payload_v3.json'                         # <- put the current payload here (or upload)
PAYLOAD_OUT  = '/content/payload_v4_labeled.json'
OVERLAY_DIR  = '/content/drive/MyDrive/hyperspectral_datasets/labeled_overlays'

import os
os.makedirs(OVERLAY_DIR, exist_ok=True)
print('DRIVE_ROOT exists:', os.path.isdir(DRIVE_ROOT))
print('OVERLAY_DIR :', OVERLAY_DIR)

## 3. Loaders — hyperspectral cubes, GT masks, RGB previews

Robust to MATLAB v5/v7 (`scipy.io.loadmat`), v7.3 HDF5 (`h5py`), and plain image previews.

In [ ]:
import numpy as np, h5py, io, base64, json, re
from pathlib import Path
from scipy.io import loadmat
from PIL import Image, ImageDraw, ImageFont

def load_any_mat(path):
    """Return the largest numeric array in a .mat file, trying v5/v7 first, then HDF5."""
    try:
        d = loadmat(path)
        arrs = [(k, v) for k, v in d.items() if not k.startswith('__') and isinstance(v, np.ndarray)]
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            arrs = [(k, np.array(f[k])) for k in f.keys() if isinstance(f[k], h5py.Dataset)]
    if not arrs: return None, None
    arrs.sort(key=lambda kv: kv[1].size, reverse=True)
    return arrs[0]  # (key, array)

def find_gt_files(root):
    """Yield (stem, gt_path) for anything with a _gt / _mask / groundtruth suffix."""
    for p in Path(root).rglob('*'):
        if not p.is_file(): continue
        low = p.name.lower()
        if p.suffix.lower() not in {'.mat', '.tif', '.tiff', '.png', '.npy'}: continue
        if any(tag in low for tag in ('_gt', '_mask', 'ground_truth', 'groundtruth')):
            stem = re.sub(r'(_gt|_mask|_ground_truth|_groundtruth).*$', '', p.stem, flags=re.I)
            yield stem, str(p)

def find_preview(root, stem):
    for p in Path(root).rglob(stem + '*.jpg'):
        if any(tag in p.name.lower() for tag in ('rgb', 'preview')) or True:
            return str(p)
    for p in Path(root).rglob(stem + '*.png'):
        return str(p)
    return None

## 4. Overlay rendering — class colors + baked legend

For every scene with a GT mask we colorize each class with a distinct color from a categorical palette, blend over the RGB preview at 55% opacity, and paint a legend strip at the bottom listing `class N — # px`.

In [ ]:
# Palette from the dashboard's category taxonomy plus extras for high-class-count GTs
PALETTE = [
    (255,122, 69), ( 79,209,255), (255,200, 87), (126,216,184), (163,224,102),
    (199,146,234), (246,134,189), ( 74, 85,104), (255, 87,133), (100,255,218),
    (147,197,253), (251,146, 60), (167,139,250), (110,231,183), (253,224, 71),
    (244,114,182), ( 96,165,250),
]

def colorize_gt(gt: np.ndarray, palette=PALETTE):
    """gt is an integer label map; 0 = background. Returns (RGB uint8 image, class_counts)."""
    gt = np.asarray(gt).astype(np.int32)
    H, W = gt.shape[-2:]
    if gt.ndim == 3: gt = gt.reshape(H, W)   # collapse a stray channel
    classes = [int(c) for c in np.unique(gt) if c != 0]
    rgb = np.zeros((H, W, 3), dtype=np.uint8)
    counts = {}
    for i, c in enumerate(classes):
        mask = gt == c
        rgb[mask] = palette[i % len(palette)]
        counts[c] = int(mask.sum())
    return rgb, classes, counts

def blend_overlay(preview_rgb: np.ndarray, gt_rgb: np.ndarray, alpha=0.55):
    """preview + gt overlay; unlabeled pixels stay untouched."""
    if preview_rgb.shape[:2] != gt_rgb.shape[:2]:
        prev = Image.fromarray(preview_rgb).resize((gt_rgb.shape[1], gt_rgb.shape[0]), Image.BILINEAR)
        preview_rgb = np.asarray(prev)
    mask = (gt_rgb.sum(-1) > 0)[..., None]
    out = preview_rgb.astype(np.float32)
    out = np.where(mask, out * (1-alpha) + gt_rgb.astype(np.float32) * alpha, out)
    return np.clip(out, 0, 255).astype(np.uint8)

def add_legend(img_rgb, classes, counts, palette=PALETTE, footer_h=None):
    """Paint a legend strip at the bottom listing 'class N — # px' in class color."""
    H, W, _ = img_rgb.shape
    per_line = max(1, W // 90)
    n_lines  = int(np.ceil(len(classes) / max(per_line, 1)))
    row_h    = 14
    footer_h = footer_h or (12 + n_lines * row_h)
    canvas   = Image.new('RGB', (W, H + footer_h), (10, 14, 26))
    canvas.paste(Image.fromarray(img_rgb), (0, 0))
    draw = ImageDraw.Draw(canvas)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 10)
    except Exception:
        font = ImageFont.load_default()
    for i, c in enumerate(classes):
        col = palette[i % len(palette)]
        r, cidx = divmod(i, per_line)
        x = 8 + cidx * (W // per_line)
        y = H + 6 + r * row_h
        draw.rectangle((x, y+1, x+10, y+11), fill=col)
        draw.text((x+14, y), f'class {c} — {counts.get(c,0)} px', fill=(230,230,230), font=font)
    return np.asarray(canvas)

def to_data_uri(rgb, quality=80):
    im = Image.fromarray(rgb)
    buf = io.BytesIO()
    im.save(buf, format='JPEG', quality=quality, optimize=True)
    return 'data:image/jpeg;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

## 5. Walk the shared folder and build overlays

Pairs every GT mask with its cube + RGB preview, generates a labeled overlay, saves it to Drive under `labeled_overlays/`, and stores the base64 into `scene.labeled_thumb`.

In [ ]:
with open(PAYLOAD_IN) as f:
    payload = json.load(f)
index = {s['stem']: s for s in payload['scenes']}

found = 0
for stem, gt_path in find_gt_files(DRIVE_ROOT):
    if stem not in index:
        # try trimming common suffixes
        trimmed = re.sub(r'(_data|_scene|_hsi|_msi|_HS_LR)$', '', stem, flags=re.I)
        if trimmed in index: stem = trimmed
    scene = index.get(stem)
    if not scene:
        continue

    # 1. load GT
    ext = Path(gt_path).suffix.lower()
    if ext == '.mat':
        _, gt = load_any_mat(gt_path)
    elif ext == '.npy':
        gt = np.load(gt_path)
    else:
        gt = np.asarray(Image.open(gt_path))
    if gt is None or gt.ndim not in (2, 3):
        continue

    # 2. load preview RGB
    prev_path = find_preview(DRIVE_ROOT, stem)
    if prev_path:
        prev = np.asarray(Image.open(prev_path).convert('RGB'))
    else:
        # synthesize a grayscale backdrop from the GT shape if no preview exists
        prev = np.zeros((gt.shape[-2], gt.shape[-1], 3), dtype=np.uint8) + 32

    # 3. render labeled overlay
    gt_rgb, classes, counts = colorize_gt(gt)
    blended = blend_overlay(prev, gt_rgb, alpha=0.55)
    labeled = add_legend(blended, classes, counts)

    # 4. save to Drive + store base64
    out_path = os.path.join(OVERLAY_DIR, stem + '_labeled.jpg')
    Image.fromarray(labeled).save(out_path, format='JPEG', quality=85, optimize=True)
    scene['labeled_thumb'] = to_data_uri(labeled, quality=80)
    scene['labeled_source'] = 'paper_gt'
    scene['labeled_classes'] = classes
    scene['labeled_class_counts'] = counts
    found += 1
    print(f'  {stem:<40} {gt.shape}  classes={len(classes)}  ->  {out_path}')

print(f'\nRebuilt {found} scenes with paper-GT overlays')

## 6. Save the updated payload and download it

Drop `payload_v4_labeled.json` into the dashboard repo, re-bake `index.html`, push. Done.

In [ ]:
payload['schema_version'] = 4
payload['overlay_method'] = {
    'kind': 'paper_ground_truth',
    'note': 'Labeled overlays come from the source-paper GT masks. Each class gets a distinct color from a categorical palette; the legend is baked into the JPEG.',
}
with open(PAYLOAD_OUT, 'w') as f:
    json.dump(payload, f, separators=(',', ':'))

print(f'Wrote {PAYLOAD_OUT}  ({os.path.getsize(PAYLOAD_OUT)/1024/1024:.2f} MB)')

from google.colab import files
files.download(PAYLOAD_OUT)

## 7. What to do with the downloaded payload

In your local checkout of `thedesignfusion/groundtruthdatasets`:

```bash
cp ~/Downloads/payload_v4_labeled.json /path/to/repo/payload_v4.json
# edit bake.py to point at payload_v4.json
python bake.py
git add index.html && git commit -m 'Swap in paper GT-labeled overlays' && git push
```

Vercel redeploys automatically; every scene with a paper GT mask now shows its true labels.